In [1]:
# ── 1. IMPORTS & REPRODUCIBILITY ───────────────────────────────
import os, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE} | AMP Enabled")
os.makedirs('../results', exist_ok=True)
os.makedirs('../models', exist_ok=True)

Device: cuda | AMP Enabled


In [2]:
# ── 2. CONSTANTS, DATA & FOCAL LOSS ────────────────────────────
MODEL_NAME    = 'roberta-base'
MAX_LEN       = 128
BATCH_SIZE    = 16 
NUM_CLASSES   = 3
N_SPLITS      = 5  # 5-Fold Ensemble

# Merge train and val to create a full 9,000-sample training set for K-Fold
df_train = pd.read_csv('../data/train.csv')
df_val   = pd.read_csv('../data/val.csv')
df_full_train = pd.concat([df_train, df_val]).reset_index(drop=True)
df_test  = pd.read_csv('../data/test.csv')

with open('../data/class_weights.json') as f:
    cw_dict = {int(k): v for k, v in json.load(f).items()}
cw_tensor = torch.tensor([cw_dict[i] for i in range(3)], dtype=torch.float).to(DEVICE)

class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(weight=alpha, reduction='none')
    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        pt = torch.exp(-ce_loss)
        return (((1 - pt) ** self.gamma) * ce_loss).mean()

criterion = FocalLoss(alpha=cw_tensor, gamma=2.0)
tokeniser = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

class TransformerDataset(Dataset):
    def __init__(self, df, tokeniser, max_len):
        texts  = df['combined_text'].fillna('').astype(str).tolist()
        labels = df['label'].tolist()
        encoding = tokeniser(texts, max_length=max_len, padding='max_length', truncation=True, return_tensors='pt', return_token_type_ids=False)
        self.input_ids = encoding['input_ids']
        self.attention_mask = encoding['attention_mask']
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {'input_ids': self.input_ids[idx], 'attention_mask': self.attention_mask[idx], 'labels': self.labels[idx]}

# We only need one test_loader for the final evaluation
test_loader = DataLoader(TransformerDataset(df_test, tokeniser, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)

In [3]:
# ── 3. STRATIFIED K-FOLD ENSEMBLE TRAINING ────────────────────
print("\n" + "=" * 60)
print(f"STARTING {N_SPLITS}-FOLD ENSEMBLE TRAINING (VANILLA RoBERTa)")
print("=" * 60)

kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
test_probas_ensemble = np.zeros((len(df_test), NUM_CLASSES))
scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

# Hardcoded optimal hyperparams for Vanilla RoBERTa
LR = 2e-5
EPOCHS = 4

for fold, (train_idx, val_idx) in enumerate(kf.split(df_full_train, df_full_train['label'])):
    print(f"\n--- Training Fold {fold + 1}/{N_SPLITS} ---")
    
    # Create Fold-Specific DataLoaders
    fold_train_df = df_full_train.iloc[train_idx]
    fold_val_df   = df_full_train.iloc[val_idx]
    
    fold_train_loader = DataLoader(TransformerDataset(fold_train_df, tokeniser, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
    fold_val_loader   = DataLoader(TransformerDataset(fold_val_df, tokeniser, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)
    
    # Initialize fresh Native RoBERTa for this fold
    model = RobertaForSequenceClassification.from_pretrained(
        MODEL_NAME, 
        num_labels=NUM_CLASSES,
        hidden_dropout_prob=0.2,
        attention_probs_dropout_prob=0.2
    ).to(DEVICE)
    
    no_decay = ['bias', 'LayerNorm.weight']
    param_groups = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
    ]
    optimizer = AdamW(param_groups, lr=LR, eps=1e-8)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * len(fold_train_loader) * EPOCHS), num_training_steps=len(fold_train_loader) * EPOCHS)
    
    best_val_f1 = 0.0
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for batch in tqdm(fold_train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                outputs = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
                loss = criterion(outputs.logits, batch['labels'].to(DEVICE))
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in fold_val_loader:
                outputs = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
                all_preds.extend(outputs.logits.argmax(1).cpu().numpy())
                all_labels.extend(batch['labels'].cpu().numpy())
                
        vl_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        print(f"  Epoch {epoch} | Val Macro F1: {vl_f1:.4f}")
        
        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            # Save the best weights for this specific fold
            torch.save(model.state_dict(), f'../models/roberta_fold_{fold+1}.pt')
            
    # ── FOLD TESTING (Accumulate Probabilities) ──
    print(f"  Extracting Test Set Probabilities for Fold {fold + 1}...")
    model.load_state_dict(torch.load(f'../models/roberta_fold_{fold+1}.pt', map_location=DEVICE))
    model.eval()
    
    fold_test_probas = []
    with torch.no_grad():
        for batch in test_loader:
            outputs = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
            fold_test_probas.extend(torch.softmax(outputs.logits, 1).cpu().numpy())
            
    # Add this fold's probabilities to the ensemble total
    test_probas_ensemble += np.array(fold_test_probas) / N_SPLITS


STARTING 5-FOLD ENSEMBLE TRAINING (VANILLA RoBERTa)

--- Training Fold 1/5 ---


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/4:   0%|          | 0/450 [00:00<?, ?it/s]

  Epoch 1 | Val Macro F1: 0.6436


Epoch 2/4:   0%|          | 0/450 [00:00<?, ?it/s]

...

  Epoch 2 | Val Macro F1: 0.6946


Epoch 3/4:   0%|          | 0/450 [00:00<?, ?it/s]

  Epoch 3 | Val Macro F1: 0.7186


Epoch 4/4:   0%|          | 0/450 [00:00<?, ?it/s]

  Epoch 4 | Val Macro F1: 0.7205
  Extracting Test Set Probabilities for Fold 5...


In [4]:
# ── 4. FINAL ENSEMBLE EVALUATION ─────────────────────────────
print("\n" + "=" * 60)
print("EVALUATING 5-FOLD ENSEMBLE ON TEST SET")
print("=" * 60)

# Extract final true labels
test_labels = []
for batch in test_loader: 
    test_labels.extend(batch['labels'].cpu().numpy())

# The ensemble prediction is the argmax of the averaged probabilities
ensemble_preds = np.argmax(test_probas_ensemble, axis=1)

test_macro_f1 = f1_score(test_labels, ensemble_preds, average='macro', zero_division=0)
test_acc = accuracy_score(test_labels, ensemble_preds)
test_auc = roc_auc_score(label_binarize(test_labels, classes=[0,1,2]), test_probas_ensemble, multi_class='ovr', average='macro')

print(f"FINAL ENSEMBLE MACRO F1 : {test_macro_f1:.4f}")
print(f"FINAL ENSEMBLE ACCURACY : {test_acc:.4f}")
print(f"FINAL ENSEMBLE ROC-AUC  : {test_auc:.4f}")

# Save Results
results = {
    'model': 'Vanilla RoBERTa (5-Fold Stratified Ensemble)',
    'macro_f1': round(float(test_macro_f1), 4),
    'accuracy': round(float(test_acc), 4),
    'roc_auc': round(float(test_auc), 4),
}
with open('../results/06_results_roberta_ensemble.json', 'w') as f:
    json.dump(results, f, indent=2)


EVALUATING 5-FOLD ENSEMBLE ON TEST SET
FINAL ENSEMBLE MACRO F1 : 0.7445
FINAL ENSEMBLE ACCURACY : 0.8840
FINAL ENSEMBLE ROC-AUC  : 0.9292
